# 04_1 — Probabilistic CPI Uncertainty & Multi-Well FWL Optimization

This notebook provides a focused deep dive into two core Field Development Planning (FDP) capabilities in `quick_pp`:
1. **Latin Hypercube Log-Level CPI Uncertainty (`run_cpi_uncertainty` / `run_log_uncertainty`)** and **Stochastic Reservoir Summary (`calc_stochastic_reservoir_summary`, `mc_volumetric_uncertainty`)**.
2. **Multi-Well Free Water Level (FWL) & Capillary Transition Zone Optimization (`optimize_fwl`, `ContactOptimizer`, `estimate_mdt_fwl`, `plot_fwl_reconciliation`)** using bundled formation pressure data (`notebooks/data/01_raw/formation_pressure_data.csv`) and multi-well logs.

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
for _cand in (Path.cwd(), Path.cwd().parent):
    if (_cand / "quick_pp" / "__init__.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quick_pp import (
    run_cpi_uncertainty,
    run_log_uncertainty,
    sample_lhs_parameters,
    mc_volumetric_uncertainty,
    calc_stochastic_reservoir_summary,
    estimate_mdt_fwl,
    predict_sw_shm,
    analyze_transition_zone,
    optimize_fwl,
    ContactOptimizer,
    plot_fwl_reconciliation,
)

## Part A: Latin Hypercube Sampling & Log-Level CPI Uncertainty (`run_cpi_uncertainty`)

In [ ]:
# Load bundled well dataset
data_path = Path("../quick_pp/machine_learning/mock_data/mock_data.parquet")
if not data_path.exists():
    data_path = Path("quick_pp/machine_learning/mock_data/mock_data.parquet")

df = pd.read_parquet(data_path).sort_values("DEPTH").reset_index(drop=True)
df["ZONES"] = np.where(df["DEPTH"] <= df["DEPTH"].median(), "ZONE_A", "ZONE_B")

# Define truncated-normal (min, max, mean, std) and uniform (min, max) parameter distributions
bounds = {
    "a": (0.85, 1.15, 1.0, 0.05),
    "m": (1.80, 2.20, 2.0, 0.08),
    "n": (1.80, 2.20, 2.0, 0.08),
    "rw": (0.03, 0.12, 0.06, 0.02),
    "rho_ma": (2.64, 2.75, 2.68, 0.03),
    "rho_sh": (2.58, 2.78, 2.68, 0.04),
}

lhs_samples = sample_lhs_parameters(bounds, n_samples=200, seed=42)
print("Sampled Latin Hypercube Parameter Summary (200 realizations):")
display(lhs_samples.describe().loc[["mean", "std", "min", "50%", "max"]].round(4))

cpi_df, reals = run_cpi_uncertainty(
    df,
    model="ssc",
    saturation_method="waxman_smits",
    n_samples=200,
    param_bounds=bounds,
    seed=42,
    return_realizations=True,
)

stoch = calc_stochastic_reservoir_summary(
    df,
    cutoffs={"VSHALE": 0.40, "PHIT": 0.05, "SWT": 0.75},
    model="ssc",
    n_samples=150,
    param_bounds=bounds,
    area_bound=(600.0, 2000.0, 1200.0, 250.0),
    volume_factor_bound=(1.15, 1.35),
    seed=42,
)
display(stoch["summary_df"])

## Part B: MDT/RFT Pressure Gradient Analysis & `ContactOptimizer`

In [ ]:
# Load bundled formation pressure data (Gas, Oil, Water legs)
pres_path = Path("data/01_raw/formation_pressure_data.csv")
if not pres_path.exists():
    pres_path = Path("notebooks/data/01_raw/formation_pressure_data.csv")

pres_df = pd.read_csv(pres_path)
oil_water_pres = pres_df[pres_df["FLUID_TYPE"].str.upper().isin(["OIL", "WATER"])].copy()

mdt_res = estimate_mdt_fwl(oil_water_pres, hc_label="OIL", water_label="WATER", unit="imperial")
print(f"Estimated MDT Oil-Water FWL: {mdt_res['fwl']:.2f} ft TVDSS | Water Grad: {mdt_res['water_gradient']:.3f} psi/ft | Oil Grad: {mdt_res['hc_gradient']:.3f} psi/ft")

# Build a 3-well synthetic appraisal dataset straddling the MDT FWL (~4970 ft TVDSS)
rng = np.random.default_rng(7)
depths_base = np.linspace(4880.0, 5040.0, 161)
wells = []
for w_idx, (w_name, fwl_shift) in enumerate([("WELL-1", 0.0), ("WELL-2", 2.5), ("WELL-3", -2.0)]):
    phit_w = np.clip(0.21 + 0.04 * np.sin((depths_base - 4880.0) / 18.0 + w_idx), 0.06, 0.30)
    perm_w = 10.0 ** (1.2 + 7.5 * (phit_w - 0.12))
    vsh_w = np.clip(0.35 - 1.1 * phit_w, 0.02, 0.45)
    rf_w = np.where(perm_w >= 80.0, 1, 2)
    shm_map = {
        1: {"a": 0.14, "b": 1.75, "swirr": 0.10},
        2: {"a": 0.24, "b": 1.45, "swirr": 0.18},
    }
    sw_clean = predict_sw_shm(
        depth=depths_base,
        fwl=mdt_res["fwl"] + fwl_shift,
        model="leverett_j",
        shm_params=shm_map,
        phit=phit_w,
        perm=perm_w,
        rock_flag=rf_w,
    )
    sw_obs = np.clip(sw_clean + rng.normal(0.0, 0.03, size=len(depths_base)), 0.08, 1.0)
    wells.append(pd.DataFrame({
        "WELL_NAME": w_name,
        "TVDSS": depths_base,
        "PHIT": phit_w,
        "PERM": perm_w,
        "VSHALE": vsh_w,
        "ROCK_FLAG": rf_w,
        "SWT": sw_obs,
    }))

multi_well_df = pd.concat(wells, ignore_index=True)

# Fit using the object-oriented ContactOptimizer
optimizer = ContactOptimizer(
    model="leverett_j",
    shm_params={
        1: {"a": 0.14, "b": 1.75, "swirr": 0.10},
        2: {"a": 0.24, "b": 1.45, "swirr": 0.18},
    },
    loss="rmse",
    pore_volume_weighted=True,
    solve_per_well=True,
)
optimizer.fit(
    multi_well_df,
    depth_col="TVDSS",
    sw_col="SWT",
    pressure_df=oil_water_pres,
    mdt_penalty_weight=0.10,
)

summary = optimizer.summary()
print("ContactOptimizer Summary:", {k: round(v, 3) if isinstance(v, float) else v for k, v in summary.items() if not isinstance(v, pd.DataFrame)})
display(summary["well_metrics"].round(3))

fig = optimizer.plot(depth_col="TVDSS", sw_col="SWT", well_col="WELL_NAME")
plt.show()